In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/"
DATASET_ID = "produccion"
TABLE_ID= "ERRORES_desgravamen_prestamos"
FECHA_PERIODO= "2025-05-30"

#PROJECT_ID = "test-proyect-468615"
#BUCKET_NAME = "data_bucket_proy"
#FOLDER_PATH= "desgravamen_prestamos/"
#DATASET_ID = "db_test"
#TABLE_ID= "desgravamen_prestamos"


# SCRIPT COMPLETO

In [ ]:
### CLIENTES DE STORAGE Y BIGQUERY
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

bucket = storage_client.bucket(BUCKET_NAME)
blob_errores = bucket.blob('data_entries/REPORTES DE ERRORES/tabla_errores2.xlsx')
df_errores= pd.read_excel(BytesIO(blob_errores.download_as_string()), sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})
df_errores.loc[df_errores['CODIGO_ERROR'].isna(), 'CODIGO_ERROR'] = ""
df_errores.loc[df_errores['IDEERROR'].isna(), 'IDEERROR'] = ""
#df_errores.drop(['IDEERROR'], axis=1, inplace=True)

schema_desgramen = [
        bigquery.SchemaField("NRO_LOTE", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("LOTES_ANTERIORES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PLAN", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("NOMBRE_DE_PLAN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_DE_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECINI_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECFIN_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN ", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SUMA_ASEGURADA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA_RECARGO", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMABRUTACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMANETACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("NOMCOMPLETO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEPATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEMATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECNACIMIENTO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR_SAS", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_PERIODO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_SEGURO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_MOVIMIENTO_NUM", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_REGISTRO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("IDEERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DETALLE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION_2", bigquery.enums.SqlTypeNames.STRING)
        ]
#### FUNCION PARA LIMPIAR Y TRANSFORMAR LA COLUMNA A UN FORMATO DE FECHA
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f' ----- Se ha creado la tabla "{table_id}" en el dataset "{dataset_id}" -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return


### LISTAR LOS ARCHIVOS QUE ESTAN DENTRO DEL BUCKET
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

### CICLO POR LA LISTA DE ARCHIVOS EXCEL
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    print(f"CARGANDO ARCHIVO: {blob.name} ..........")
    file = blob.download_as_string()
    df_desgravamen= pd.read_excel(BytesIO(file), sheet_name='Exportar Hoja de Trabajo', dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str,
                                                                             'IDELOTE': str, 'CODIGO ERROR': str,
                                                                             'SUMA ASEGURADA':str, 'TASA': str,
                                                                             'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})
    ###### LIMPIEZA Y TRANSFORMACIONES

    # Colocar _ en los espacio de los nombres de las columnas
    df_desgravamen.columns = (df_desgravamen.columns.str.strip()
                                                    .str.upper()  # opcional: todo en mayúsculas
                                                    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                              )
    # Borrar Columnas repetidas

    df_desgravamen.drop(['IDELOTE','IDEDET_1'], axis=1, inplace=True)
    df_desgravamen= df_desgravamen.rename(columns={'DESCRIPCION_ERROR':'DESCRIPCION_ERROR_SAS'})

    # Imputar valores nulos
    df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].replace(r"^\s*$", None, regex=True)
    df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].where(df_desgravamen["MONEDA"].isin(["USD", "SOL"]), "SIN DATO")
    df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
    df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'
    # Cambiar el sepador de decimales
    df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
    df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
    df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
    df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
    df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

    #Cambiar el tipo de datos de las columnas
    df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
    df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
    df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
    df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
    df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
    df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
    df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
    df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
    df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
    df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce").astype('float64')
    df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce").astype('float64')
    df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce").astype('float64')
    df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce").astype('float64')
    df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce").astype('float64')
    df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
    df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
    df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
    df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
    df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
    df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
    df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
    df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
    df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
    df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
    df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<30000]   #Eliminar las filas con valores atipicos muy altos
    df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)
    df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
    # Columna para diferenciar el periodo de reporte de errores
    df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
    # Crear columna evaluando condiciones en el contenido de otras columnas
    df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)

    #Extraer datos de la la Trama Original
    df_desgravamen["TIPO_SEGURO"] = df_desgravamen["LINEA_TRAMA"].str[:3]
    df_desgravamen["TIPO_MOVIMIENTO_NUM"] = df_desgravamen["LINEA_TRAMA"].str[48]
    df_desgravamen["TIPO_REGISTRO"] = df_desgravamen["LINEA_TRAMA"].str[43:45]

    # Cruzar con la Tabla de errores
    df_desgravamen= df_desgravamen.merge(df_errores, on='CODIGO_ERROR', how='left')

    # Guardar tabla en BigQuery
    Guardar_en_BigQuery(df_desgravamen, DATASET_ID, TABLE_ID, schema_desgramen)
    print(f"##### EL ARCHIVO: {blob.name} SE HA GUARDADO CORRECTAMENTE EN BIGQUERY #####")



CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte01.xlsx ..........
 ----- Se ha creado la tabla "ERRORES_desgravamen_prestamos" en el dataset "produccion" -----
##### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte01.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY #####
CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte02.xlsx ..........
##### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte02.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY #####
CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte03.xlsx ..........
##### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 -

--------------

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blob_errores = bucket.blob('data_entries/REPORTES DE ERRORES/tabla_errores2.xlsx')

In [ ]:
df_errores= pd.read_excel(BytesIO(blob_errores.download_as_string()), sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})
df_errores.loc[df_errores['CODIGO_ERROR'].isna(), 'CODIGO_ERROR'] = ""
df_errores.loc[df_errores['IDEERROR'].isna(), 'IDEERROR'] = ""
#df_errores.drop(['IDEERROR'], axis=1, inplace=True)
df_errores.head(3)

,IDEERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,0974,1068,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,CONFIGURACION SAS,ANULAR MASIVAS
1,0005,0003,ERROR EN CARGA DE TRAMA,ERROR EN CARGA DE TRAMA,ANALISIS EMISOR,NaN
2,1158,1155,TRAMA REPETIDA O DUPLICADA,TRAMA REPETIDA O DUPLICADA,CONFIGURACION SAS,ANULAR MASIVAS


In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

In [ ]:
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    #file = blob.download_as_string()
    #excel_file = pd.ExcelFile(file)
    print(blob.name)
    #print(excel_file.sheet_names)

data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte01.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte02.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte03.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte04.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte05.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte06.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte07.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte08.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. 

In [ ]:
#blob_consuer = bucket.blob('desgravamen_prestamos/desgravamen - consuer.csv')
#file = blob_consuer.download_as_string()
#df_desgravamen= pd.read_csv(BytesIO(file), dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str, 'IDELOTE': str, 'CODIGO ERROR': str})

In [ ]:
print(blobs_excels[3].name)
file = blobs_excels[3].download_as_string()
df_desgravamen= pd.read_excel(BytesIO(file), sheet_name='Exportar Hoja de Trabajo', dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str,
                                                                                           'IDELOTE': str, 'CODIGO ERROR': str,
                                                                                           'SUMA ASEGURADA':str, 'TASA': str,
                                                                                           'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})


data_entries/REPORTES DE ERRORES/DESGRAVAMEN/1. Prestamos/30-mayo-2025/446 - Desgravamen + Desempleo - BBVA_pte03.xlsx


In [ ]:
df_desgravamen.head(3)

,NRO LOTE,LOTES ANTERIORES,FECHA CARGA,CODIGO PRODUCTO,PRODUCTO,CODIGO PLAN,NOMBRE DE PLAN,COD DE CERTIFICADO,fecini_alta_CERTIFICADO,fecfin_alta_CERTIFICADO,IDEDET,TIPO MOVIMIENTO,FEC. INICIO,FEC. FIN,MONEDA,SUMA ASEGURADA,TASA,TASA RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE DE ARCHIVO,LINEA TRAMA,IDELOTE,IDEDET_1,ORIGEN ERROR,CODIGO ERROR,DESCRIPCION ERROR
0,290309,NaN,2019-11-05 17:36:20,3827.0,Desgravamen + Desempleo BBVA,80908.0,Plan Conticasa,00110766344000006395,NaT,NaT,336399462,Inclusion,16/10/2019,02/11/2032,SOL,1390050,"0,52","13,02",0,0,ROLANDO RAMON,CARO,HARTER,19520909.0,2.0,29247950,20100130204_0157001_20191105_001.TXT,903001107663440000063950011076639960000730301P...,290309,336399462,Error canal,1077,LA EDAD DE PERMANENCIA DEL ASEGURADO (FEC. FIN...
1,290309,NaN,2019-11-05 17:36:20,3827.0,Desgravamen + Desempleo BBVA,80908.0,Plan Conticasa,00110220134001147074,NaT,NaT,336398961,Inclusion,17/10/2019,30/10/2036,SOL,159223,"0,52","50,52",0,0,FORTUNATO ABRAHAM,HUARCAYA,CHAVA,19560515.0,2.0,29503286,20100130204_0157001_20191105_001.TXT,903001102201340011470740011022014960083793501P...,290309,336398961,Error canal,1077,LA EDAD DE PERMANENCIA DEL ASEGURADO (FEC. FIN...
2,291180,NaN,2019-11-07 10:45:47,NaN,NaN,NaN,NaN,NaN,NaT,NaT,336832102,Sin Movimiento,14/10/2019,14/11/2019,SOL,0,0,0,"905,83","879,45",NaN,NaN,NaN,NaN,NaN,NaN,20100130204_0157001_20191106_002.TXT,903001103605040002855720011010623020022658101P...,291180,336832102,Error canal,1155,Trama repetida o duplicada


In [ ]:
df_desgravamen.columns = (
    df_desgravamen.columns
    .str.strip()  # quitar espacios al inicio/fin
    .str.upper()  # opcional: todo en mayúsculas
    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)
df_desgravamen.drop(['IDELOTE','IDEDET_1'], axis=1, inplace=True)
df_desgravamen= df_desgravamen.rename(columns={'DESCRIPCION_ERROR':'DESCRIPCION_ERROR_SAS'})
#df_desgravamen.drop(['IDEDET_1','ORIGEN_ERROR','UNNAMED__32'], axis=1, inplace=True)
#df_desgravamen= df_desgravamen.rename(columns={'IDELOTE': 'LINEA_TRAMA','CODIGO_ERROR':'ORIGEN_ERROR', 'DESCRIPCION_ERROR': 'CODIGO_ERROR'})

In [ ]:
df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].replace(r"^\s*$", None, regex=True)
df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].where(df_desgravamen["MONEDA"].isin(["USD", "SOL"]), "SIN DATO")

In [ ]:
df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'

In [ ]:
df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

In [ ]:
df_desgravamen.head()

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR_SAS
0,290309,,2019-11-05 17:36:20,3827.0,Desgravamen + Desempleo BBVA,80908.0,Plan Conticasa,00110766344000006395,NaT,NaT,336399462,Inclusion,16/10/2019,02/11/2032,SOL,1390050,0.52,13.02,0,0,ROLANDO RAMON,CARO,HARTER,19520909.0,2.0,29247950,20100130204_0157001_20191105_001.TXT,903001107663440000063950011076639960000730301P...,Error canal,1077,LA EDAD DE PERMANENCIA DEL ASEGURADO (FEC. FIN...
1,290309,,2019-11-05 17:36:20,3827.0,Desgravamen + Desempleo BBVA,80908.0,Plan Conticasa,00110220134001147074,NaT,NaT,336398961,Inclusion,17/10/2019,30/10/2036,SOL,159223,0.52,50.52,0,0,FORTUNATO ABRAHAM,HUARCAYA,CHAVA,19560515.0,2.0,29503286,20100130204_0157001_20191105_001.TXT,903001102201340011470740011022014960083793501P...,Error canal,1077,LA EDAD DE PERMANENCIA DEL ASEGURADO (FEC. FIN...
2,291180,,2019-11-07 10:45:47,NaN,NaN,NaN,NaN,NaN,NaT,NaT,336832102,Sin Movimiento,14/10/2019,14/11/2019,SOL,0,0,0,905.83,879.45,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0157001_20191106_002.TXT,903001103605040002855720011010623020022658101P...,Error canal,1155,Trama repetida o duplicada
3,291180,,2019-11-07 10:45:47,NaN,NaN,NaN,NaN,NaN,NaT,NaT,336833744,Sin Movimiento,28/10/2019,28/11/2019,SOL,0,0,0,135.48,131.53,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0157001_20191106_002.TXT,903001108315540003649440011017817020046998301P...,Error canal,1155,Trama repetida o duplicada
4,291180,,2019-11-07 10:45:47,3827.0,Desgravamen + Desempleo BBVA,80908.0,Plan Conticasa,00110831514000929011,2019-06-03,2036-12-22,336833449,Renovacion,30/10/2019,30/11/2019,SOL,0,0,0,127.4,123.69,NaN,NaN,NaN,NaN,SIN DATO,NaN,20100130204_0157001_20191106_002.TXT,903001108315140009290110011018722020036432301P...,Error canal,1150,EL ALTA DEL NRO. CERTIFICADO EXTERNO: 00110831...


In [ ]:
df_desgravamen[df_desgravamen['MONEDA'].isna()].head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR_SAS


In [ ]:
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [ ]:
df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
#df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce")
#df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce")
#df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce")
#df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce")
#df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce")
df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce").astype('float64')
df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce").astype('float64')
df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce").astype('float64')
df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce").astype('float64')
df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce").astype('float64')
df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
#df_desgravamen['IDEERROR'] = df_desgravamen['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")
#df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else np.nan)
df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

In [ ]:
df_desgravamen['CODIGO_PRODUCTO'].value_counts()

,count
CODIGO_PRODUCTO,
0,3184
3070,2848
3827,713


In [ ]:
#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

In [ ]:
df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<30000]   #Eliminar las filas con valores atipicos muy altos
df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)    # Crear colunma evaluando condiciones en el contenido de otras columnas

In [ ]:
df_desgravamen["TIPO_SEGURO"] = df_desgravamen["LINEA_TRAMA"].str[:3]
df_desgravamen["TIPO_MOVIMIENTO_NUM"] = df_desgravamen["LINEA_TRAMA"].str[48]
df_desgravamen["TIPO_REGISTRO"] = df_desgravamen["LINEA_TRAMA"].str[43:45]

In [ ]:
df_desgravamen[df_desgravamen['CODIGO_ERROR']== ''].head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR_SAS,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR,TIPO_SEGURO,TIPO_MOVIMIENTO_NUM,TIPO_REGISTRO
72,303834,,2019-12-03 12:41:23,3070,Desgravamen BBVA,60085,Plan Mi Vivienda,00110108874000040048,NaT,NaT,346387254,Renovacion,2019-11-29,2019-12-29,SOL,0.0,0.0,0.0,50.46,48.99,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20191203_002.TXT,906001101088740000400480011014274020077019901P...,Error Rimac,,[**ERROR**]Errores al validar la p¿a: [1] No s...,2019-12-03,2025-05-30,Mes corriente,906,4,01
73,303834,,2019-12-03 12:41:23,3070,Desgravamen BBVA,60085,Plan Mi Vivienda,00110108874000040048,NaT,NaT,346387254,Renovacion,2019-11-29,2019-12-29,SOL,0.0,0.0,0.0,50.46,48.99,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20191203_002.TXT,906001101088740000400480011014274020077019901P...,Error Rimac,,[**ERROR**]Errores al validar la p¿a: [1] No s...,2019-12-03,2025-05-30,Mes corriente,906,4,01
74,303834,,2019-12-03 12:41:23,3070,Desgravamen BBVA,60085,Plan Mi Vivienda,00110108874000040048,NaT,NaT,346387254,Renovacion,2019-11-29,2019-12-29,SOL,0.0,0.0,0.0,50.46,48.99,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20191203_002.TXT,906001101088740000400480011014274020077019901P...,Error Rimac,,[**ERROR**]Errores al validar la p¿a: [1] No s...,2019-12-03,2025-05-30,Mes corriente,906,4,01


In [ ]:
#df_desgravamen= df_desgravamen.merge(df_errores, on='IDEERROR', how='left')
#df_desgravamen.drop(['IDEERROR'], axis=1, inplace=True)
df_desgravamen= df_desgravamen.merge(df_errores, on='CODIGO_ERROR', how='left')
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR_SAS,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR,TIPO_SEGURO,TIPO_MOVIMIENTO_NUM,TIPO_REGISTRO,IDEERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,290309,,2019-11-05 17:36:20,3827,Desgravamen + Desempleo BBVA,80908,Plan Conticasa,00110766344000006395,NaT,NaT,336399462,Inclusion,2019-10-16,2032-11-02,SOL,1390050.0,0.52,13.02,0.00,0.00,ROLANDO RAMON,CARO,HARTER,1952-09-09,2.0,29247950,20100130204_0157001_20191105_001.TXT,903001107663440000063950011076639960000730301P...,Error canal,1077,LA EDAD DE PERMANENCIA DEL ASEGURADO (FEC. FIN...,2019-11-05,2025-05-30,Mes corriente,903,1,01,NaN,NaN,NaN,NaN,NaN
1,290309,,2019-11-05 17:36:20,3827,Desgravamen + Desempleo BBVA,80908,Plan Conticasa,00110220134001147074,NaT,NaT,336398961,Inclusion,2019-10-17,2036-10-30,SOL,159223.0,0.52,50.52,0.00,0.00,FORTUNATO ABRAHAM,HUARCAYA,CHAVA,1956-05-15,2.0,29503286,20100130204_0157001_20191105_001.TXT,903001102201340011470740011022014960083793501P...,Error canal,1077,LA EDAD DE PERMANENCIA DEL ASEGURADO (FEC. FIN...,2019-11-05,2025-05-30,Mes corriente,903,1,01,NaN,NaN,NaN,NaN,NaN
2,291180,,2019-11-07 10:45:47,0,nan,0,nan,nan,NaT,NaT,336832102,Sin Movimiento,2019-10-14,2019-11-14,SOL,0.0,0.00,0.00,905.83,879.45,nan,nan,nan,NaT,SIN DATO,nan,20100130204_0157001_20191106_002.TXT,903001103605040002855720011010623020022658101P...,Error canal,1155,Trama repetida o duplicada,2019-11-06,2025-05-30,Mes corriente,903,4,01,1158,TRAMA REPETIDA O DUPLICADA,TRAMA REPETIDA O DUPLICADA,CONFIGURACION SAS,ANULAR MASIVAS


In [ ]:
df_desgravamen['DESCRIPCION_ERROR'].value_counts()

,count
DESCRIPCION_ERROR,
ERROR EN CARGA DE TRAMA,3363
VALIDACIONES CONSECUENCIAS,294
VALIDACIONES CONSECUENCIA ACSEL E,235
CAMPOS EN BLANCO,214
TRAMA REPETIDA O DUPLICADA,181
PAGO PENDIENTE DE CORREGIR,43
ERROR EN ALTA,41
SIN POLIZA,16
ERROR EDAD,8


In [ ]:
df_desgravamen["DESCRIPCION_ERROR"] = df_desgravamen["DESCRIPCION_ERROR"].astype(str)
df_desgravamen["DETALLE"] = df_desgravamen["DETALLE"].astype(str)
df_desgravamen["SOLUCION"] = df_desgravamen["SOLUCION"].astype(str)
df_desgravamen["SOLUCION_2"] = df_desgravamen["SOLUCION_2"].astype(str)

In [ ]:
df_desgravamen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6745 entries, 0 to 6744
Data columns (total 42 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   NRO_LOTE                 6745 non-null   int64         
 1   LOTES_ANTERIORES         6745 non-null   object        
 2   FECHA_CARGA              6745 non-null   datetime64[ns]
 3   CODIGO_PRODUCTO          6745 non-null   int64         
 4   PRODUCTO                 6745 non-null   object        
 5   CODIGO_PLAN              6745 non-null   int64         
 6   NOMBRE_DE_PLAN           6745 non-null   object        
 7   COD_DE_CERTIFICADO       6745 non-null   object        
 8   FECINI_ALTA_CERTIFICADO  1608 non-null   datetime64[ns]
 9   FECFIN_ALTA_CERTIFICADO  1608 non-null   datetime64[ns]
 10  IDEDET                   6745 non-null   int64         
 11  TIPO_MOVIMIENTO          6745 non-null   object        
 12  FEC__INICIO              6743 non-

In [ ]:
TABLE_ID= "ERRORES_desgravamen_prestamos_temp"
schema_desgramen = [
        bigquery.SchemaField("NRO_LOTE", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("LOTES_ANTERIORES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PLAN", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("NOMBRE_DE_PLAN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_DE_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECINI_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECFIN_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN ", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SUMA_ASEGURADA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA_RECARGO", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMABRUTACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMANETACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("NOMCOMPLETO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEPATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEMATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECNACIMIENTO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR_SAS", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_PERIODO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_SEGURO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_MOVIMIENTO_NUM", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_REGISTRO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("IDEERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DETALLE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION_2", bigquery.enums.SqlTypeNames.STRING)
    ]
Guardar_en_BigQuery(df_desgravamen, DATASET_ID, TABLE_ID, schema_desgramen)

----- REGISTROS AGREGADOS CORRECTAMENTE EN: ERRORES_desgravamen_prestamos_temp -------
